In [1]:
import cv2
import numpy as np

In [2]:
def split_image_with_overlap(image_path, overlap_ratio=0.4, output_prefix="split"):
    """
    Splits a wide panorama image into two overlapping parts.

    Parameters:
        image_path (str): Path to the input panorama image.
        overlap_ratio (float): Fraction of image width that overlaps (0.0–1.0).
        output_prefix (str): Prefix for output image filenames.

    Returns:
        (part1, part2): The two overlapping image parts (numpy arrays).
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Error: Image not found or invalid path.")

    height, width, _ = img.shape
    print(f"Original image size: {width}x{height}")

    # Define split positions
    overlap_width = int(width * overlap_ratio)
    split_point = int(width * 0.7)  # first image covers up to 70% of width

    # Ensure bounds
    split_point = min(max(split_point, overlap_width), width - overlap_width)

    # Define slices
    part1 = img[:, :split_point]
    part2 = img[:, split_point - overlap_width:]

    # Save output images
    part1_path = f"{output_prefix}_part1.jpg"
    part2_path = f"{output_prefix}_part2.jpg"

    cv2.imwrite(part1_path, part1)
    cv2.imwrite(part2_path, part2)

    print(f"Saved {part1_path} and {part2_path}")
    print(f"Overlap width: {overlap_width}px (~{overlap_ratio*100:.0f}%)")

    return part1, part2

In [3]:
split_image_with_overlap("images/high-tatras-in-winter.jpg")

Original image size: 1920x1080
Saved split_part1.jpg and split_part2.jpg
Overlap width: 768px (~40%)


(array([[[ 73,  16,   1],
         [ 73,  16,   1],
         [ 73,  16,   1],
         ...,
         [101,  27,   0],
         [101,  27,   0],
         [101,  27,   0]],
 
        [[ 73,  16,   1],
         [ 73,  16,   1],
         [ 73,  16,   1],
         ...,
         [102,  28,   0],
         [102,  28,   0],
         [102,  28,   0]],
 
        [[ 73,  16,   1],
         [ 73,  16,   1],
         [ 73,  16,   1],
         ...,
         [102,  28,   0],
         [102,  28,   0],
         [102,  28,   0]],
 
        ...,
 
        [[204, 182, 171],
         [203, 181, 170],
         [205, 180, 170],
         ...,
         [228, 216, 210],
         [226, 214, 208],
         [224, 212, 206]],
 
        [[204, 182, 171],
         [203, 181, 170],
         [205, 180, 170],
         ...,
         [225, 215, 208],
         [223, 213, 206],
         [222, 212, 205]],
 
        [[203, 181, 170],
         [203, 181, 170],
         [204, 179, 169],
         ...,
         [221, 211, 204],
  

In [4]:
def stitch_images(img1_path, img2_path, output_path="panorama_result.jpg"):
    # Load images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)

    if img1 is None or img2 is None:
        raise ValueError("Error: One or both image paths are invalid.")

    # Convert images to grayscale
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Create ORB detector to find keypoints and descriptors
    orb = cv2.ORB_create(5000)

    # Detect keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)

    # Match features using Brute Force matcher with Hamming distance
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    # Sort matches by distance (best matches first)
    matches = sorted(matches, key=lambda x: x.distance)

    # Use only good matches
    good_matches = matches[:int(len(matches) * 0.15)]

    # Draw matches (optional for visualization)
    matched_img = cv2.drawMatches(img1, kp1, img2, kp2, good_matches, None, flags=2)
    cv2.imwrite("matches.jpg", matched_img)

    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Find homography matrix
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

    # Warp the second image to align with the first
    width = img1.shape[1] + img2.shape[1]
    height = max(img1.shape[0], img2.shape[0])
    panorama = cv2.warpPerspective(img2, H, (width, height))
    panorama[0:img1.shape[0], 0:img1.shape[1]] = img1

    # Crop black regions (simple method)
    gray_panorama = cv2.cvtColor(panorama, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray_panorama, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x, y, w, h = cv2.boundingRect(contours[0])
    cropped_panorama = panorama[y:y+h, x:x+w]

    # Save and display result
    cv2.imwrite(output_path, cropped_panorama)
    print(f"Panorama saved as {output_path}")

    return cropped_panorama

In [5]:
stitch_images("split_part1.jpg", "split_part2.jpg")

Panorama saved as panorama_result.jpg


array([[[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       [[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       [[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       ...,

       [[204, 182, 171],
        [203, 181, 170],
        [205, 180, 170],
        ...,
        [211, 202, 199],
        [217, 205, 203],
        [219, 207, 205]],

       [[204, 182, 171],
        [203, 181, 170],
        [205, 180, 170],
        ...,
        [216, 206, 206],
        [219, 209, 209],
        [220, 210, 210]],

       [[203, 181, 170],
        [203, 181, 170],
        [204, 179, 169],
        ...,
        [232, 222, 222],
        [233, 223, 223],
        [234, 224, 224]]

In [6]:
def stitch_perspective(img1_path, img2_path, output_path="panorama_perspective.jpg"):
    """
    Stitches two images that may have different perspectives using SIFT feature matching.
    """

    # Load images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)

    if img1 is None or img2 is None:
        raise ValueError("Error: Could not load one or both images.")

    # Convert to grayscale
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Use SIFT (Scale-Invariant Feature Transform)
    sift = cv2.SIFT_create()

    # Detect and compute keypoints/descriptors
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    # Match features using FLANN-based matcher
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1, des2, k=2)

    # Apply Lowe’s ratio test to filter good matches
    good = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good.append(m)

    print(f"Number of good matches: {len(good)}")

    if len(good) < 10:
        raise RuntimeError("Not enough matches found to compute homography.")

    # Extract point coordinates from good matches
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    # Compute homography using RANSAC
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

    # Warp second image to align with first
    height, width, _ = img1.shape
    warped_img2 = cv2.warpPerspective(img2, H, (width + img2.shape[1], height))

    # Blend the two images (simple overlay)
    warped_img2[0:img1.shape[0], 0:img1.shape[1]] = img1

    # Crop black borders
    gray = cv2.cvtColor(warped_img2, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x, y, w, h = cv2.boundingRect(contours[0])
    result = warped_img2[y:y+h, x:x+w]

    # Save result
    cv2.imwrite(output_path, result)
    print(f"Panorama saved as {output_path}")

    return result

In [7]:
stitch_perspective("split_part1.jpg", "split_part2.jpg")

Number of good matches: 1284
Panorama saved as panorama_perspective.jpg


array([[[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       [[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       [[ 73,  16,   1],
        [ 73,  16,   1],
        [ 73,  16,   1],
        ...,
        [115,  32,   0],
        [115,  32,   0],
        [115,  32,   0]],

       ...,

       [[204, 182, 171],
        [203, 181, 170],
        [205, 180, 170],
        ...,
        [211, 202, 199],
        [217, 205, 203],
        [219, 207, 205]],

       [[204, 182, 171],
        [203, 181, 170],
        [205, 180, 170],
        ...,
        [216, 206, 206],
        [219, 209, 209],
        [220, 210, 210]],

       [[203, 181, 170],
        [203, 181, 170],
        [204, 179, 169],
        ...,
        [232, 222, 222],
        [233, 223, 223],
        [234, 224, 224]]

In [8]:
def stitch_with_opencv_stitcher(img1_path, img2_path, output_path="panorama_stitcher_cropped.jpg"):
    """
    Automatically stitches two images using OpenCV's built-in Stitcher class.
    Handles perspective alignment, exposure, blending, and auto-cropping.
    """

    # Load images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)
    if img1 is None or img2 is None:
        raise ValueError("Error: Could not load one or both images.")

    # Create Stitcher instance
    # MODE_PANORAMA works well for images with rotation or perspective change
    try:
        stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)
    except AttributeError:
        # For older OpenCV versions
        stitcher = cv2.createStitcher(cv2.Stitcher_PANORAMA)

    # Stitch images
    status, stitched = stitcher.stitch([img1, img2])
    if status != cv2.Stitcher_OK:
        raise RuntimeError(f"Stitching failed (error code: {status})")

    print("Stitching successful. Cropping result...")

    # Convert to grayscale for mask detection
    gray = cv2.cvtColor(stitched, cv2.COLOR_BGR2GRAY)

    # Threshold to create binary mask where non-black pixels are white
    _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    # Find contours of the non-black area
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise RuntimeError("No contours found for cropping.")

    # Get bounding rectangle for the largest contour
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    cropped = stitched[y:y+h, x:x+w]

    # Optional: refine crop by trimming black pixels near edges
    gray_crop = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    _, thresh_crop = cv2.threshold(gray_crop, 1, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh_crop)
    x2, y2, w2, h2 = cv2.boundingRect(coords)
    final_cropped = cropped[y2:y2+h2, x2:x2+w2]

    # Save and return the final result
    cv2.imwrite(output_path, final_cropped)
    print(f"Final panorama saved as {output_path}")

    return final_cropped

In [9]:
stitch_with_opencv_stitcher("split_part1.jpg", "split_part2.jpg")

Stitching successful. Cropping result...
Final panorama saved as panorama_stitcher_cropped.jpg


array([[[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[ 74,  16,   0],
        [ 74,  16,   0],
        [ 74,  16,   0],
        ...,
        [116,  32,   0],
        [116,  32,   0],
        [116,  32,   0]],

       ...,

       [[200, 177, 167],
        [200, 176, 167],
        [200, 176, 166],
        ...,
        [209, 200, 195],
        [214, 203, 199],
        [214, 204, 200]],

       [[200, 176, 167],
        [200, 176, 167],
        [200, 176, 167],
        ...,
        [213, 204, 202],
        [215, 207, 204],
        [215, 206, 205]],

       [[200, 176, 167],
        [200, 176, 167],
        [200, 176, 166],
        ...,
        [225, 216, 214],
        [225, 216, 215],
        [226, 217, 215]]